# Testing the pure-C engine's GPU backend on a Colab GPU

This notebook validates the OpenCL backend (`gpu.c`) on a real discrete GPU.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU** (or any GPU).

**Build the binary first, on your own machine** (it is not built here -- Colab has no
copy of the repo):

```
powershell -File tools\build_gpu.ps1 -Only colab     # Windows
make cross-gpu                                        # Linux/macOS
```

That produces `build/transformer-linux-x86_64-gpu`. Upload it in the cell below.

It needs **no CUDA toolkit and no OpenCL SDK**: the driver is opened with `dlopen`
at runtime and its entry points resolved by name, which is exactly why a Windows
box with no NVIDIA GPU can cross-compile a working GPU binary for this machine.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Install an OpenCL ICD loader

Colab ships the NVIDIA driver (which *provides* OpenCL via
`libnvidia-opencl.so.1`) but usually not the vendor-neutral **loader**
`libOpenCL.so.1`, nor the ICD registration file that points the loader at
NVIDIA. Both are one line each.

If you skip this cell the binary still runs -- `gpu.c` falls back to opening
`libnvidia-opencl.so.1` directly -- but installing the loader is the supported
path and lets `clinfo` corroborate what the engine reports.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ocl-icd-libopencl1 clinfo > /dev/null
!mkdir -p /etc/OpenCL/vendors && echo libnvidia-opencl.so.1 > /etc/OpenCL/vendors/nvidia.icd
!clinfo -l

## 3. Upload the cross-compiled binary

In [ ]:
from google.colab import files
up = files.upload()          # pick build/transformer-linux-x86_64-gpu
import os
BIN = './' + list(up)[0]
os.chmod(BIN, 0o755)
print('uploaded:', BIN)

## 4. Does the engine see the GPU?

Expect `status: READY` and an NVIDIA device name. If it says **NO USABLE
DEVICE**, that is a correct and supported state -- every matmul just runs on the
CPU -- and the message tells you what is missing.

In [ ]:
!$BIN gpu-info

## 5. Correctness FIRST

The full smoke test. Part **[30]** is the one that matters here: it runs the
forward matmul and both backward matmuls on deliberately *ragged* shapes (not
multiples of the kernel's 16-wide tile or 64-column block -- the boundary case a
tiled GEMM classically gets wrong) and requires the GPU to agree with the CPU.
It also checks the backward kernels **accumulate** into pre-seeded gradients
rather than overwriting them.

A fast wrong answer is not a speedup. This must print
`Output validation: PASSED` before any timing below is worth reading.

In [ ]:
!$BIN 2>&1 | tail -n 30

## 6. Then speed

CPU vs GPU on the GEMM shapes SmolLM2-135M actually uses. Note the Colab CPU is
typically only 2 vCPUs, so the CPU column here is much weaker than a real
desktop -- the honest comparison for "should I train on Colab" is this GPU column
against your own machine's `gpu-bench` CPU column.

Reference points measured on an Intel UHD 730 integrated GPU (24 EUs):
**~62 GFLOP/s**, which *lost* to a 10-core i5-14400 at ~170-240 GFLOP/s. A T4
should be a different story entirely.

In [ ]:
!$BIN gpu-bench 5

## 7. End-to-end A/B on a real workload

`--no-gpu` forces the CPU path in the *same binary*, which is what makes this an
honest comparison rather than two different builds.

This trains a small model from scratch on a tiny corpus -- no checkpoint upload
needed. (Model `.bin` files are git-ignored and ~621 MB; upload one to your
Drive and point `--resume` at it if you want to benchmark the real 135M.)

In [ ]:
%%bash -s "$BIN"
BIN=$1
mkdir -p data
python3 - <<'PY'
words = "the quick brown fox jumps over a lazy dog while stars wheel above cold rivers".split()
import random; random.seed(1)
with open('data/bench.txt','w') as f:
    for _ in range(400):
        f.write(' '.join(random.choice(words) for _ in range(40)) + '\n')
PY
echo '--- GPU ---'
time $BIN llama-train data/bench.txt --steps 20 --d-model 256 --layers 4 --heads 8 \
     --ctx 128 --out /tmp/g.bin --tok /tmp/g.tok --gpu
echo '--- CPU ---'
time $BIN llama-train data/bench.txt --steps 20 --d-model 256 --layers 4 --heads 8 \
     --ctx 128 --out /tmp/c.bin --tok /tmp/c.tok --no-gpu

## What to take away

- **`Output validation: PASSED` with part [30] green** is the real deliverable:
  the GPU kernels agree with the CPU on ragged shapes and accumulate correctly.
- The GPU is used only where it wins by design. Single-token decode (`M=1`) is
  memory-bandwidth-bound and deliberately stays on the CPU (`TF_GPU_MIN_M`), so
  do not expect generation to speed up -- training and prefill are the targets.
- Tune without rebuilding: `TF_GPU_MIN_M` and `TF_GPU_MIN_MACS` are env vars.
- Known headroom: every operand is uploaded on every call, because caching
  weight *contents* cannot be done safely without an explicit immutability hint
  from the model code (`attention.c` passes activations as the B operand). That
  is the next optimization, and it is worth roughly 2x.